In [ ]:
!pip install allensdk

Looking in indexes: https://bbpteam.epfl.ch/repository/devpi/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 27.0 MB/s eta 0:00:00 MB/s eta 0:00:01:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 46.9 MB/s eta 0:00:0031m38.9 MB/s eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 332.0/332.0 kB 59.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 56.4 MB/s eta 0:00:00m eta 0:00:010:00:01
  Using cached https://bbpteam.epfl.ch/repository/devpi/root/pypi/%2Bf/7a0/a56cef15fd158/pandas-1.5.3-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (12.1 MB)
  Using cached https://bbpteam.epfl.ch/repository/devpi/root/pypi/%2Bf/4c0/ff64b06b10e35/scipy-1.10.1-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (34.4 MB)
  Using cached future-0.18.3-py3-none-any.whl
  Using cached https://bbpteam.epfl.ch/repository/devpi/root/pypi/%2Bf/ccc/fdd665f0a24fc/requests_toolbelt-1.0.0-py2.py3-none-any.whl (54 kB)
     ━━━━━━━━━━━━━

In [ ]:
import pandas as pd
from analysis_neuro import terminology as terms

In [ ]:
import os
import shutil

import numpy as np
import pandas as pd

from allensdk.brain_observatory.ecephys.ecephys_project_cache import EcephysProjectCache

data_directory = '/gpfs/bbp.cscs.ch/project/proj148/home/dictus/ecephys_cache_dir' # must be updated to a valid directory in your filesystem

manifest_path = os.path.join(data_directory, "manifest.json")

cache = EcephysProjectCache.from_warehouse(manifest=manifest_path)

metrics = cache.get_unit_analysis_metrics_by_session_type('brain_observatory_1.1')

In [ ]:
#experiment = pd.read_csv("siegle_validation_data.csv")

FS = metrics['waveform_duration'] < 0.5
metrics = metrics.assign(spiking_class=['FS' if at else 'RS' for at in FS])

experiment = metrics[['ecephys_structure_acronym', 'spiking_class', 'g_osi_dg', 'firing_rate', 'firing_rate_dg', ]]
experiment

In [ ]:
units = cache.get_units()
units

In [ ]:
xyz = units[[
    'anterior_posterior_ccf_coordinate', 'dorsal_ventral_ccf_coordinate',
    'left_right_ccf_coordinate']
].values
from voxcell.nexus.voxelbrain import Atlas
atlas = Atlas.open("/gpfs/bbp.cscs.ch/project/proj148/atlas/VISp/20220512/")
rmap = atlas.load_region_map()
region_ids = atlas.load_data("brain_regions").lookup(xyz)
acronyms = [rmap.get(id_, 'acronym') for region_id in region_ids]

def _get_layer(acronym):
    if acronym[-1] in ['123456']:
        return f'L{acronym[-1]}'
    if acronym[-2] == '6':
        return f'L{acronym[-2:]}'
    return None

layers = [_get_layer(acronym) for acronym in acronyms]
units['layer'] = layers

In [4]:
from analysis_neuro import stimuli

observations = pd.DataFrame({
    terms.FIRING_RATE: experiment['firing_rate'],
    terms.VISUAL_STIMULUS: stimuli.ALLEN_BRAIN_OBSERVATORY + 'gray',
    terms.REGION: experiment['ecephys_structure_acronym'],
    terms.SPIKING_CLASS: experiment['spiking_class'],
    terms.CITATION: 'siegle_survey_2019',
    terms.DATASET: 'Siegle2019'
})
observations.to_csv("analysis-neuro/analysis_neuro/analyses/data/siegle-spontaneous-2019.csv", index=False)

In [8]:
import numpy as np
osi = pd.DataFrame({
    terms.ORIENTATION_SELECTIVITY: experiment['g_osi_dg'],
    terms.VISUAL_STIMULUS: stimuli.ALLEN_BRAIN_OBSERVATORY + 'drifting gratings',
    terms.TEMPORAL_FREQUENCY: 'optimal',
    terms.REGION: experiment['ecephys_structure_acronym'],
    terms.SPIKING_CLASS: experiment['spiking_class'],
    terms.CITATION: 'siegle_survey_2019',
    terms.DATASET: 'Siegle2019'
})
assert not np.any(np.isnan(osi[terms.ORIENTATION_SELECTIVITY]))
osi.to_csv("analysis-neuro/analysis_neuro/analyses/data/siegle-osi-2019.csv", index=False)

In [ ]:
optimal = pd.DataFrame({
    terms.FIRING_RATE: experiment['firing_rate_dg'],
    terms.VISUAL_STIMULUS: stimuli.ALLEN_BRAIN_OBSERVATORY + 'drifting gratings',
    terms.TEMPORAL_FREQUENCY: 'optimal',
    terms.STIM_ORIENTATION: 'optimal',
    terms.REGION: experiment['ecephys_structure_acronym'],
    terms.SPIKING_CLASS: experiment['spiking_class'],
    terms.CIRATION: 'siegle_survey_2019',
    terms.DATASET: 'Siegle2019'
})
optimal.to_csv("analysis-neuro/analysis_neuro/analyses/data/siegle-optimal-2019.csv")